# Complete RAG Application: Retrieval, Augmentation, and Generation

**Stack:** Python · LangChain · PyPDF · Recursive Character Text Splitter · Hugging Face Embeddings · ChromaDB · Google Gemini

This notebook builds a practical PDF-based Retrieval-Augmented Generation (RAG) application in clearly separated stages.

## The three RAG stages — keep these names clear

1. **RETRIEVAL** — search the vector database and fetch the most relevant document chunks.
2. **AUGMENTATION** — combine the retrieved context with the user's question to construct an augmented prompt.
3. **GENERATION** — send the augmented prompt to the LLM (Gemini) and generate the answer.

The notebook also includes **document ingestion** (loading, chunking, embedding, and storage), which happens before a user asks a question.

> **Important:** Keep your PDF file in the same folder as this notebook, or provide its full path in the PDF configuration cell. You will enter your Gemini API key securely when prompted; do not paste API keys into notebook code or upload them to GitHub.

## 0. RAG workflow

### A. Document ingestion (one-time setup for a document)

`PDF → Text extraction → Chunking → Embedding generation → ChromaDB storage`

### B. Question answering (runs for each user query)

`User question → Query embedding → RETRIEVAL → AUGMENTATION → GENERATION → Final answer`

- **ChromaDB** stores chunk text, metadata, and vectors, and performs vector retrieval.
- **The application** assembles retrieved text and the question into an augmented prompt.
- **Gemini** generates the response from that prompt.

In this notebook, the three RAG stages are explicitly named in headings, code comments, and the final pipeline function.

## 1. Install the required libraries

Run this cell once in your notebook environment. If you install or upgrade packages while Jupyter is running, restart the kernel if imports fail.

In [2]:
%pip install -qU langchain langchain-community langchain-text-splitters langchain-chroma chromadb pypdf sentence-transformers langchain-huggingface langchain-google-genai

## 2. Import libraries and configure the project

We use:
- `PyPDFLoader` to extract text and page metadata from a PDF.
- `RecursiveCharacterTextSplitter` to split text while respecting natural text boundaries where possible.
- `HuggingFaceEmbeddings` to create vectors for document chunks and user questions.
- `Chroma` as the persistent vector store.
- `ChatGoogleGenerativeAI` for the **GENERATION** stage.

In [ ]:
from pathlib import Path
import os
import getpass

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

# Upload PDF file to Google Colab
from google.colab import files

uploaded = files.upload()

# Get uploaded PDF filename
pdf_filename = list(uploaded.keys())[0]

# Set PDF path
PDF_PATH = Path("/content") / pdf_filename

print("PDF uploaded successfully!")
print("PDF Name:", pdf_filename)
print("PDF Path:", PDF_PATH)
print("File Exists:", PDF_PATH.exists())

/tmp/ipykernel_17777/4084211478.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


: 

## 3. Configure the Gemini API key

Get an API key from Google AI Studio. Enter it in the secure prompt below. It will be stored only in this running notebook session as an environment variable.

Do not write the key directly into a code cell, commit it to Git, or share the notebook with the key embedded.

In [ ]:
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Gemini API key: ")

print("Gemini API key is configured for this session.")

## 4. Load the PDF and extract text

This is the **document-loading / text-extraction** part of ingestion. `PyPDFLoader` returns a list of LangChain `Document` objects. Each object contains page text (`page_content`) and metadata such as the source file and page number.

In [ ]:
# Load PDF Document

loader = PyPDFLoader(str(PDF_PATH))

pages = loader.load()

print("Total Pages:", len(pages))

# Display first page metadata
print("\nPage Metadata:")
print(pages[0].metadata)

# Display extracted text
print("\nExtracted Text:")
print(pages[0].page_content[:1000])

## 5. Chunk the document — Recursive Chunking

We use `RecursiveCharacterTextSplitter`.

- `chunk_size=800`: target maximum chunk length in **characters** (not tokens).
- `chunk_overlap=120`: target overlap in **characters** between neighboring chunks.
- The splitter tries separators in this order by default: paragraph break (`\\n\\n`), newline (`\\n`), space, and then character-level splitting.

The splitter tries to keep natural text units together, but a chunk can still be split when it exceeds the target size.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    length_function=len,
    add_start_index=True
)

chunks = text_splitter.split_documents(pages)

print("Total chunks created:", len(chunks))
print("\nFirst chunk preview:")
print(chunks[0].page_content if chunks else "No chunks were created.")

print("\nFirst chunk metadata:")
print(chunks[0].metadata if chunks else "No metadata available.")

## 6. Generate embeddings

An embedding model converts text into a numerical vector. We use the same embedding model for document chunks and incoming user questions so their vectors are comparable.

`all-MiniLM-L6-v2` commonly produces 384-dimensional vectors. The exact output shape is checked below rather than assumed.

In [ ]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

# Demonstrate embedding generation for one chunk.
sample_embedding = embedding_model.embed_query(chunks[0].page_content)

print("Embedding dimensions:", len(sample_embedding))
print("First 10 vector values:", sample_embedding[:10])

## 7. Store the chunks and embeddings in ChromaDB

This is the **vector storage** part of ingestion.

ChromaDB stores:
- the original chunk text,
- its embedding vector,
- metadata such as source and page number.

The collection is recreated each time this cell runs so that rerunning the notebook does not accidentally append duplicate chunks from the same PDF. This clears only this notebook's named collection in the specified local ChromaDB folder.

In [ ]:
import chromadb

# Create/open a persistent local ChromaDB client.
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))

# Recreate this notebook's collection to avoid duplicate records on reruns.
try:
    chroma_client.delete_collection(name=COLLECTION_NAME)
except Exception:
    pass

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name=COLLECTION_NAME,
    client=chroma_client,
    ids=[f"chunk-{i}" for i in range(len(chunks))]
)

print("Chunks stored in ChromaDB:", vector_store._collection.count())
print("Collection name:", COLLECTION_NAME)
print("Persistent database folder:", CHROMA_DIR)

## 8. Inspect records stored in ChromaDB

Use `peek()` to inspect a small sample and `get()` to fetch stored records. These methods inspect stored data; they are not the semantic retrieval step used to answer a question.

In [ ]:
print("Sample records using peek():")
print(vector_store._collection.peek())

print("\nNumber of stored records using get():")
stored_records = vector_store.get()
print(len(stored_records["ids"]))

print("\nFirst stored ID:")
print(stored_records["ids"][0] if stored_records["ids"] else "No records")

print("\nFirst stored document preview:")
print(stored_records["documents"][0][:500] if stored_records["documents"] else "No documents")

# PART 2 — QUESTION ANSWERING

## 9. Enter a user question and create its query embedding

When a user asks a question, the same embedding model converts that question into a query vector. This vector is used to search the stored chunk vectors.

This cell demonstrates query embedding separately so you can see that the query is also converted into a numerical vector.

In [ ]:
question = "What is the main topic discussed in this document?"

query_embedding = embedding_model.embed_query(question)

print("User question:", question)
print("Query embedding dimensions:", len(query_embedding))
print("First 10 query vector values:", query_embedding[:10])

## 10. RETRIEVAL — search and fetch relevant chunks

**Retrieval** means finding and fetching the document chunks most relevant to the user's question.

We use:
- `similarity_search(query, k=3)` to fetch the top 3 relevant chunks.
- `similarity_search_with_score(query, k=3)` to fetch chunks with Chroma's distance scores.

**Important about scores:** Chroma's `similarity_search_with_score()` returns a distance value for the configured distance metric. For the usual Chroma L2 distance, a **smaller distance means a closer match**. Do not automatically interpret this distance as a cosine-similarity percentage. The ranking is what matters here.

In [ ]:
# RETRIEVAL: find the 3 most relevant chunks for the user's question.
retrieved_docs = vector_store.similarity_search(question, k=3)

print("Number of chunks retrieved:", len(retrieved_docs))
for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\n--- Retrieved Chunk {i} ---")
    print("Metadata:", doc.metadata)
    print("Text:", doc.page_content)

### 10.1 Inspect retrieval distance scores

This second retrieval call shows the distance returned by ChromaDB. The score is useful for comparing results from the same collection and configuration. Its scale depends on the distance metric and embedding configuration.

In [ ]:
# RETRIEVAL: return relevant chunks together with their distance scores.
retrieved_with_scores = vector_store.similarity_search_with_score(question, k=3)

for i, (doc, distance) in enumerate(retrieved_with_scores, start=1):
    print(f"\n--- Result {i} ---")
    print("Distance score:", distance, "(lower is closer for L2 distance)")
    print("Metadata:", doc.metadata)
    print("Text:", doc.page_content[:700])

### 10.2 Experiment with different questions

Change the question and rerun this cell. Observe how the retrieved chunks change according to the meaning of the query.

In [ ]:
test_questions = [
    "What is the main topic of this document?",
    "Which important facts or concepts are explained?",
    "What conclusion does the document provide?"
]

for test_question in test_questions:
    print("\n" + "=" * 90)
    print("QUESTION:", test_question)
    test_results = vector_store.similarity_search(test_question, k=2)
    for rank, doc in enumerate(test_results, start=1):
        print(f"\nRetrieved chunk {rank} | metadata: {doc.metadata}")
        print(doc.page_content[:500])

## 11. AUGMENTATION — build the augmented prompt

**Augmentation** means combining the user's original question with the context retrieved from the document.

The LLM does not search ChromaDB by itself in this notebook. Our Python code takes the retrieved chunks, formats them as context, and inserts that context and the user's question into a prompt template.

We also include an instruction to say when the answer is not present in the supplied context.

In [ ]:
# AUGMENTATION: combine the retrieved chunk text into one context string.
context = "\n\n".join(
    f"Source {i}: {doc.page_content}"
    for i, doc in enumerate(retrieved_docs, start=1)
)

prompt_template = ChatPromptTemplate.from_template(
    """You are a helpful assistant answering questions using the supplied document context.

Instructions:
- Answer using the context provided below.
- Do not invent facts or use unsupported information.
- If the context does not contain the answer, clearly say:
  "I could not find that information in the provided document."
- Give a clear, direct answer.

DOCUMENT CONTEXT:
{context}

USER QUESTION:
{question}

ANSWER:"""
)

augmented_prompt = prompt_template.invoke({
    "context": context,
    "question": question
})

print("AUGMENTED PROMPT (what will be sent to Gemini):\n")
print(augmented_prompt.to_string())

## 12. GENERATION — send the augmented prompt to Gemini

**Generation** is the stage where the LLM reads the augmented prompt and generates the final answer.

The model receives the question together with the retrieved document context. It does not receive the entire PDF unless the entire PDF happened to be retrieved as context.

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

# GENERATION: Gemini generates an answer from the augmented prompt.
response = llm.invoke(augmented_prompt)

print("GENERATED ANSWER:\n")
print(response.content)

# PART 3 — COMPLETE END-TO-END RAG PIPELINE

## 13. Create the complete RAG function

This function clearly separates all three stages:

1. **RETRIEVAL:** search ChromaDB and fetch the top-K relevant chunks.
2. **AUGMENTATION:** combine those chunks with the user's question in a prompt.
3. **GENERATION:** send the augmented prompt to Gemini and return the answer.

The function also returns the retrieved chunks and distance scores so you can inspect why the system produced its answer.

In [ ]:
def rag_pipeline(user_question: str, k: int = 3):
    """
    End-to-end PDF RAG pipeline.

    RETRIEVAL:
        Search ChromaDB and fetch the top-k relevant chunks.

    AUGMENTATION:
        Combine retrieved chunk text with the user's question.

    GENERATION:
        Send the augmented prompt to Gemini and generate an answer.
    """

    if not user_question.strip():
        raise ValueError("Please enter a non-empty question.")

    # 1. RETRIEVAL
    retrieved_results = vector_store.similarity_search_with_score(
        user_question,
        k=k
    )

    retrieved_documents = [doc for doc, distance in retrieved_results]

    if not retrieved_documents:
        return {
            "answer": "No relevant document chunks were retrieved.",
            "retrieved_chunks": [],
            "distance_scores": [],
            "augmented_prompt": None
        }

    # 2. AUGMENTATION
    retrieved_context = "\n\n".join(
        f"Source {i}: {doc.page_content}"
        for i, doc in enumerate(retrieved_documents, start=1)
    )

    final_prompt = prompt_template.invoke({
        "context": retrieved_context,
        "question": user_question
    })

    # 3. GENERATION
    llm_response = llm.invoke(final_prompt)

    return {
        "answer": llm_response.content,
        "retrieved_chunks": retrieved_documents,
        "distance_scores": [distance for doc, distance in retrieved_results],
        "augmented_prompt": final_prompt.to_string()
    }

## 14. Test the complete RAG application

Ask a question that should be answerable from your PDF. The output displays the answer, retrieved chunks, and ChromaDB distance scores.

In [ ]:
user_question = "What are the key points explained in this document?"

result = rag_pipeline(user_question, k=3)

print("USER QUESTION:")
print(user_question)

print("\nRETRIEVAL — CHUNKS FETCHED:")
for i, doc in enumerate(result["retrieved_chunks"], start=1):
    print(f"\nChunk {i} | distance: {result['distance_scores'][i-1]}")
    print("Metadata:", doc.metadata)
    print(doc.page_content)

print("\nGENERATION — FINAL ANSWER:")
print(result["answer"])

## 15. Test a question that may not be answered by the PDF

Try asking for information that is clearly unrelated to your PDF. The prompt instructs Gemini to say when the retrieved context does not contain the answer.

**Note:** Retrieval can still return the nearest available chunks even when the question is unrelated. A production RAG system should add a calibrated relevance threshold or a separate retrieval-evaluation step; top-K retrieval alone does not guarantee that a relevant answer exists.

In [ ]:
unrelated_question = "What is the current population of the Moon?"

unrelated_result = rag_pipeline(unrelated_question, k=3)

print("QUESTION:", unrelated_question)
print("\nANSWER:")
print(unrelated_result["answer"])

## 16. Ask your own questions

Edit the question and run the cell whenever you want to query your PDF.

In [ ]:
my_question = "Write your question about the PDF here."

if my_question.strip() == "Write your question about the PDF here.":
    print("Edit my_question with a question about your PDF, then run this cell.")
else:
    my_result = rag_pipeline(my_question, k=3)
    print("Answer:")
    print(my_result["answer"])

# Final recap — remember the correct RAG terminology

| Stage | What happens | Main component/code |
|---|---|---|
| Document ingestion | Load PDF and extract text | `PyPDFLoader` |
| Chunking | Split text into manageable chunks | `RecursiveCharacterTextSplitter` |
| Embedding generation | Convert chunks into vectors | `HuggingFaceEmbeddings` |
| Vector storage | Store text, vectors, and metadata | `Chroma` |
| **RETRIEVAL** | Search and fetch relevant chunks | `similarity_search()` / `similarity_search_with_score()` |
| **AUGMENTATION** | Combine retrieved context and user question into a prompt | `ChatPromptTemplate` and Python formatting |
| **GENERATION** | Generate an answer from the augmented prompt | `ChatGoogleGenerativeAI.invoke()` |

## The three stages in one line

**RETRIEVAL** = Find and fetch relevant chunks.

**AUGMENTATION** = Add retrieved context to the user's question.

**GENERATION** = Send the augmented prompt to the LLM and generate the answer.

## Troubleshooting

- **PDF not found:** update `PDF_PATH` to the correct filename or full path.
- **Gemini authentication error:** verify your Google AI Studio API key and that the selected model is available to your account.
- **Package import error after installation:** restart the Jupyter kernel and run the notebook from the beginning.
- **No text extracted:** the PDF may be scanned images; OCR is needed before normal text chunking.
- **Unexpected retrieval results:** try a different question, inspect the retrieved chunks, and experiment with chunk size, overlap, and `k`.
- **Slow first embedding run:** the Hugging Face model may need to download the first time.

**Security:** Never share your API key or commit it to a public GitHub repository.